In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from cmdstanpy import CmdStanModel

import arviz as az

C:\Users\Josh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the 2026 dataset: https://memory.psych.upenn.edu/Data_Archive#2026

In [ ]:
# Get data
data = pd.read_csv('data/Exp3_AllData.csv', header=0, sep=',')
data.head()
data = data[~np.isnan(data["Output_Rank"])]

# Convert to ints
data["Listnum"] = data["Listnum"].astype(int)
data["Length"] = data["Length"].astype(int)
data["Serial_Pos_Encoded"] = data["Serial_Pos_Encoded"].astype(int)
data["Output_Rank"] = data["Output_Rank"].astype(int)

# Filter for only spatial and varied data
data = data[data["Order_Type"] == 'Spatial']
data = data[data["List_Type"] == 'Varied']

# Only use a fraction of the data
train_percent = 0.10
train_n = int(data.shape[0]*train_percent)
data = data.sample(n=train_n)

data_dict = {
    # Sizes
    "N": data.shape[0],
    "S": data["Uniqueid"].nunique(),
    # IDs
    "subject_index": data["Uniqueid"].astype("category").cat.codes + 1,
    # Experiment
    "spatial_input_order": data.Spatial_Input_Order.values,
    # Results
    "spatial_recall_order": data.Spatial_Recall_Order.values,
    #"output_rank": data.Output_Rank.values,
    "rt": data.Initial_RT_Time.values,
}

In [8]:
# Compile model
model = CmdStanModel(stan_file="stan/spatial_varied_model.stan")

22:51:41 - cmdstanpy - INFO - compiling stan file C:\Users\Josh\Box\Documents\COGS-4210_Cognitive_Modeling\Repo\project\stan\spatial_varied_model.stan to exe file C:\Users\Josh\Box\Documents\COGS-4210_Cognitive_Modeling\Repo\project\stan\spatial_varied_model.exe
22:52:01 - cmdstanpy - INFO - compiled model executable: C:\Users\Josh\Box\Documents\COGS-4210_Cognitive_Modeling\Repo\project\stan\spatial_varied_model.exe


In [9]:
# Sample model
fit = model.sample(data=data_dict, chains=4, iter_sampling=2500, iter_warmup=1000)

model.generate_quantities

# Display sampling diagnostics
print(fit.diagnose())

summary = fit.summary()

# Ensure all R hat values are <=1.01
thresh = 1.01
num = 0
num_past = 0
r_hat_sum = 0
for r_hat in summary['R_hat']:
    num += 1
    r_hat_sum += r_hat
    if r_hat > thresh:
        num_past += 1
print("Number of", num, "R_hat Values Past Threshold:", num_past)

22:52:58 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:07<00:19, 146.31it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:08<00:15, 168.63it/s, (Warmup)]



chain 1:  29%|██▊       | 1000/3500 [00:08<00:14, 170.56it/s, (Sampling)]

chain 1:  31%|███▏      | 1100/3500 [00:09<00:13, 172.86it/s, (Sampling)]



chain 1:  34%|███▍      | 1200/3500 [00:09<00:13, 174.70it/s, (Sampling)]


chain 1:  37%|███▋      | 1300/3500 [00:10<00:12, 174.59it/s, (Sampling)]


chain 1:  40%|████      | 1400/3500 [00:11<00:12, 173.40it/s, (Sampling)]


chain 1:  43%|████▎     | 1500/3500 [00:11<00:11, 172.76it/s, (Sampling)]




chain 1:  46%|████▌     | 1600/3500 [00:12<00:10, 175.77it/s, (Sampling)]





chain 1:  49%|████▊     | 1700/3500 [00:12<00:10, 172.81it/s, (Sampling)]


chain 1:  51%|█████▏    | 1800/3500 [00:13<00:09, 172.24it/s, (Sampling)]


chain 1:  54%|█████▍    | 1900/3500 [00:14<00:0


22:53:23 - cmdstanpy - INFO - CmdStan done processing.
22:53:23 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'spatial_varied_model.stan', line 50, column 8 to column 61)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'spatial_varied_model.stan', line 50, column 8 to column 61)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'spatial_varied_model.stan', line 50, column 8 to column 61)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'spatial_varied_model.stan', line 50, column 8 to column 61)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'spatial_varied_model.stan', line 50, column 8 to column 61)
Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'spatial_varied_model.stan', line 50, column 8 to column 61)
	Exc


Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
2 of 10000 (0.02%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete.

Number of 7 R_hat Values Past Threshold: 0


In [11]:
# Display how good R_hat values are
print("Average R_hat Value:", r_hat_sum / num)
print("R_hat score:", (1.01 - (r_hat_sum / num)) * 10000, "(100 is best)")

# Display sampling data
display(summary)

draws_df = fit.draws_pd()
display(draws_df.head())

#["sigma_recall", "correct_alpha", "correct_beta", "mu_rt", "sigma_rt", "gamma_rt"]
#idata = az.from_cmdstanpy(fit)
#az.plot_trace(idata, var_names=["sigma_recall"], legend=True, compact=True)
#plt.tight_layout()

Average R_hat Value: 1.0004371428571428
R_hat score: 95.62857142857206 (100 is best)


,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-3214.690000,0.026605,1.718230,1.607210,-3218.010000,-3214.390000,-3212.490000,4290.02,6399.00,80.9223,1.00109
sigma_recall,3.332100,0.000672,0.073550,0.072627,3.213080,3.330770,3.455820,12019.80,7296.50,226.7300,1.00045
correct_alpha,7.173250,0.014551,1.228830,1.188450,5.443580,7.023340,9.419470,8147.56,6034.94,153.6870,1.00025
correct_beta,13.851100,0.023985,1.886950,1.869150,11.017200,13.675300,17.194300,6497.07,6689.20,122.5540,1.00040
mu_rt,3.339460,0.000475,0.041267,0.041114,3.271640,3.339830,3.406430,7565.37,7470.09,142.7050,1.00029
sigma_rt,1.055690,0.000223,0.023629,0.023646,1.017610,1.055400,1.094910,11302.70,7161.43,213.2030,1.00005
gamma_rt,-0.013127,0.000143,0.012542,0.012440,-0.033836,-0.013029,0.007357,7648.45,7573.82,144.2720,1.00053


,chain__,iter__,draw__,lp__,accept_stat__,stepsize__,treedepth__,n_leapfrog__,divergent__,energy__,sigma_recall,correct_alpha,correct_beta,mu_rt,sigma_rt,gamma_rt
0,1.0,1.0,1.0,-3214.0798,0.872348,0.437867,3.0,7.0,0.0,3218.6798,3.278097,8.254718,13.711069,3.296446,1.044901,-0.019176
1,1.0,2.0,2.0,-3213.4503,0.967326,0.437867,2.0,3.0,0.0,3215.0128,3.280101,8.209942,13.515858,3.322229,1.044693,-0.022755
2,1.0,3.0,3.0,-3213.9552,0.954474,0.437867,3.0,7.0,0.0,3214.8742,3.394261,5.579100,14.699345,3.369544,1.059139,-0.016207
3,1.0,4.0,4.0,-3212.8265,0.977501,0.437867,3.0,7.0,0.0,3214.5272,3.268456,7.557023,14.267260,3.334959,1.054821,-0.022176
4,1.0,5.0,5.0,-3214.8373,0.934528,0.437867,3.0,7.0,0.0,3215.5021,3.353092,7.671734,17.883219,3.331752,1.064880,-0.003370
